# Sentiment Analysis on Amazon Reviews

## File Handling in Python

In [ ]:
reviews = []
targets = []

with open("amazon_cells_labelled.txt", "r") as file:
  for line in file:
    # line = line.split("\t")
    # line[1] = line[1].strip()
    # print(repr(line))
    review, target = line.strip().split("\t")
    reviews.append(review)
    targets.append(int(target))

In [ ]:
reviews[:2]  # Checking first 2 reviews

['So there is no way for me to plug it in here in the US unless I go by a converter.',
 'Good case, Excellent value.']

In [ ]:
targets[:2]  # Checking first 2 targets

[0, 1]

In [ ]:
targets.count(1)  # 500 positive reviews

500

In [ ]:
targets.count(0)  # 500 negative reviews

500

## Preprocessing

In [ ]:
import nltk

In [ ]:
nltk.download('stopwords') # Download stopwords from NLTK package
nltk.download('wordnet')  # Download Lemmatizer
nltk.download('punkt_tab')  # Download Tokenizer Rules


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
stop_words = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [ ]:
len(stop_words)

198

In [ ]:
lemmatizer.lemmatize("amazing", pos='v')  # Parts of Speech: n - noun, v - verb, a - adjective

'amaze'

In [ ]:
import re

def preprocess(text):

  # Convert review into lowercase
  text = text.lower()

  # Remove punctuation and numbers
  text = re.sub(r'[^a-z\s]', '', text)  # Replace everything except a-z and single space with empty string

  # Tokenize: split reviews into individual words
  tokens = word_tokenize(text)

  # Remove stop words: Examples of stop words - "the", "an", ... They don't carry any setiment, so remove it
  tokens = [token for token in tokens if token not in stop_words]

  # Lemmatize: Convert tokens to its verb form
  tokens = [lemmatizer.lemmatize(token, pos='v') for token in tokens]

  return " ".join(tokens)  # Put together the tokens back to sentence format

In [ ]:
sample_review = "Amazing products. Delivered promptly. 10/10"
preprocess(sample_review)

'amaze products deliver promptly'

In [ ]:
cleaned_reviews = []

for review in reviews:  # Loop through all the reviews and save the processed review in a new list: `cleaned_reviews`
  cleaned_reviews.append(preprocess(review))

In [ ]:
cleaned_reviews[:2]

['way plug us unless go converter', 'good case excellent value']

## Train-test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(cleaned_reviews,
                                                    targets,
                                                    test_size=0.5,
                                                    stratify=targets)

In [ ]:
len(X_train) == len(y_train)

True

In [ ]:
len(X_test) == len(y_test)

True

In [ ]:
y_train.count(1)

250

In [ ]:
y_train.count(0)

250

Same distribution is maintained after split because of the `stratify` parameter in train-test split

## Feature Extraction

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer()

In [ ]:
X_train_features = vectorizer.fit_transform(X_train)

In [ ]:
X_train_features.shape

(500, 952)

In [ ]:
vectorizer.get_feature_names_out()  # Features extracted by the TF-IDF Vectorizer

array(['abhor', 'ability', 'able', 'absolutel', 'absolutely', 'accept',
       'acceptable', 'access', 'accessory', 'accidentally', 'accompany',
       'accord', 'activate', 'activesync', 'actually', 'ad', 'adapter',
       'adapters', 'adorable', 'advertise', 'advise', 'ago', 'allot',
       'allow', 'almost', 'along', 'also', 'aluminum', 'always', 'amaze',
       'amazon', 'ample', 'another', 'answer', 'ant', 'antiglare',
       'anyone', 'anything', 'anyway', 'apart', 'apartment', 'apparently',
       'appeal', 'appearance', 'applifies', 'area', 'argue', 'around',
       'arrive', 'ask', 'aspect', 'atleast', 'att', 'attack', 'audio',
       'autoanswer', 'available', 'avoid', 'away', 'awesome', 'awful',
       'awkward', 'back', 'background', 'bad', 'balance', 'bar', 'barely',
       'bargain', 'basement', 'basic', 'batteries', 'battery', 'beat',
       'beautiful', 'behing', 'believe', 'belt', 'bend', 'best', 'bethe',
       'better', 'beware', 'big', 'biggest', 'bill', 'black',
  

## Model Training

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
multinomial_nb = MultinomialNB()

In [ ]:
model = Pipeline([
    ('feature_extractor', TfidfVectorizer()),
    ('classifier', multinomial_nb)
])

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'classifier__alpha': [0.2, 0.5, 1, 1.2, 2, 3, 5]}

grid_search = GridSearchCV(model, param_grid, refit=True, cv=5)
grid_search

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('feature_extractor', TfidfVectorizer()),
                                       ('classifier', MultinomialNB())]),
             param_grid={'classifier__alpha': [0.2, 0.5, 1, 1.2, 2, 3, 5]})

In [ ]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('feature_extractor', TfidfVectorizer()),
                                       ('classifier', MultinomialNB())]),
             param_grid={'classifier__alpha': [0.2, 0.5, 1, 1.2, 2, 3, 5]})

In [ ]:
best_model = grid_search.best_estimator_
best_model

Pipeline(steps=[('feature_extractor', TfidfVectorizer()),
                ('classifier', MultinomialNB(alpha=0.5))])

## Train Performance

In [ ]:
y_train_pred = best_model.predict(X_train)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98       250
           1       0.96      1.00      0.98       250

    accuracy                           0.98       500
   macro avg       0.98      0.98      0.98       500
weighted avg       0.98      0.98      0.98       500



## Test Performance

In [ ]:
y_test_pred = best_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.80      0.73      0.76       250
           1       0.75      0.82      0.78       250

    accuracy                           0.77       500
   macro avg       0.78      0.77      0.77       500
weighted avg       0.78      0.77      0.77       500

